# TerraSpectra — Day 6: Reproducible Train / Validation / Test Data Split

**Project:** Hyperspectral Crop Disease Forecasting  
**Day:** 6 — 18 November 2026  
**Task:** Prepare a reproducible data-loading and sample-splitting pipeline.

## Important data note
The supplied `pos_Plant21.csv` contains annotated pixel coordinates, not verified healthy/diseased class labels. This notebook extracts the 204-band spectra at those coordinates and creates reproducible **feature subsets only**. It does **not** create class labels, train a classifier, or claim disease-detection performance. Verified labels are required before supervised model training.

## Inputs
Upload either:
- `hyperspectral_day04_preprocessed.npy` (preferred, if Day 4 output is available), **or**
- `REFLECTANCE_2047.hdr` and `REFLECTANCE_2047.dat` (the notebook applies the same per-band Z-score normalization used in Day 2),

and `pos_Plant21.csv`.

## Outputs
The notebook saves `X_train.npy`, `X_val.npy`, `X_test.npy`, coordinate CSVs, and a split report under `TerraSpectra_Day06/` and creates a ZIP archive.

## 1. Install and import dependencies

In [ ]:
!pip -q install spectral scikit-learn

import os
import zipfile
import numpy as np
import pandas as pd
from google.colab import files
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
EXPECTED_SHAPE = (512, 512, 204)
print("Dependencies imported successfully.")

## 2. Upload the input files

In [ ]:
uploaded = files.upload()
print("Uploaded files:")
for name in uploaded:
    print(" -", name)

print("\nAvailable NPY cubes:", [f for f in os.listdir('/content') if f.endswith('.npy')])

## 3. Load the hyperspectral cube

In [ ]:
# Prefer the Day 4 preprocessed cube when it is available.
day4_path = "/content/hyperspectral_day04_preprocessed.npy"

if os.path.exists(day4_path):
    cube = np.load(day4_path).astype(np.float32)
    cube_source = "Day 4 preprocessed cube"
else:
    # Reproduce the Day 2 loading and per-band Z-score normalization.
    hdr_path = "/content/REFLECTANCE_2047.hdr"
    dat_path = "/content/REFLECTANCE_2047.dat"
    if not (os.path.exists(hdr_path) and os.path.exists(dat_path)):
        raise FileNotFoundError(
            "Upload hyperspectral_day04_preprocessed.npy OR both "
            "REFLECTANCE_2047.hdr and REFLECTANCE_2047.dat."
        )

    import spectral
    image = spectral.open_image(hdr_path)
    original_cube = np.asarray(image.load(), dtype=np.float32)
    band_mean = original_cube.mean(axis=(0, 1), keepdims=True)
    band_std = original_cube.std(axis=(0, 1), keepdims=True)
    band_std[band_std == 0] = 1.0
    cube = ((original_cube - band_mean) / band_std).astype(np.float32)
    cube_source = "Original ENVI data with Day 2 per-band Z-score normalization"

print("Cube source:", cube_source)
print("Cube shape:", cube.shape)
print("Cube dtype:", cube.dtype)
print("All values finite:", bool(np.isfinite(cube).all()))

assert cube.shape == EXPECTED_SHAPE, f"Expected {EXPECTED_SHAPE}, got {cube.shape}"
assert np.isfinite(cube).all(), "Cube contains NaN or infinite values."

## 4. Load and validate annotated coordinates

In [ ]:
csv_path = "/content/pos_Plant21.csv"
if not os.path.exists(csv_path):
    raise FileNotFoundError("Upload pos_Plant21.csv.")

# The supplied CSV has no header row and stores row, column coordinates.
coordinates = pd.read_csv(csv_path, header=None, names=["row", "col"])
coordinates["row"] = pd.to_numeric(coordinates["row"], errors="raise").astype(int)
coordinates["col"] = pd.to_numeric(coordinates["col"], errors="raise").astype(int)

before = len(coordinates)
coordinates = coordinates.drop_duplicates(subset=["row", "col"]).reset_index(drop=True)
removed_duplicates = before - len(coordinates)

inside = (
    coordinates["row"].between(0, cube.shape[0] - 1)
    & coordinates["col"].between(0, cube.shape[1] - 1)
)
if not inside.all():
    raise ValueError("Some coordinates fall outside the hyperspectral cube.")

print("Input coordinate rows:", before)
print("Duplicate coordinates removed:", removed_duplicates)
print("Unique valid coordinates:", len(coordinates))
display(coordinates.head())

## 5. Extract 204-band spectral features

In [ ]:
rows = coordinates["row"].to_numpy()
cols = coordinates["col"].to_numpy()

# Each coordinate becomes one sample; each spectral band becomes one feature.
X = cube[rows, cols, :].astype(np.float32)
feature_names = [f"band_{i+1}" for i in range(X.shape[1])]

print("Feature matrix shape:", X.shape)
print("Samples:", X.shape[0])
print("Spectral features per sample:", X.shape[1])
print("All feature values finite:", bool(np.isfinite(X).all()))

assert X.ndim == 2 and X.shape[1] == 204
assert np.isfinite(X).all()

## 6. Create reproducible 70 / 15 / 15 splits

In [ ]:
# First reserve 30% for validation + testing.
X_train, X_temp, coord_train, coord_temp = train_test_split(
    X, coordinates, test_size=0.30, random_state=RANDOM_STATE, shuffle=True
)

# Split the reserved 30% equally into validation and test.
X_val, X_test, coord_val, coord_test = train_test_split(
    X_temp, coord_temp, test_size=0.50, random_state=RANDOM_STATE, shuffle=True
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)
print("Total:", len(X_train) + len(X_val) + len(X_test))

## 7. Verify split integrity

In [ ]:
train_set = set(map(tuple, coord_train[["row", "col"]].to_numpy()))
val_set = set(map(tuple, coord_val[["row", "col"]].to_numpy()))
test_set = set(map(tuple, coord_test[["row", "col"]].to_numpy()))

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)
assert len(X_train) + len(X_val) + len(X_test) == len(X)
for name, array in [("train", X_train), ("validation", X_val), ("test", X_test)]:
    assert array.shape[1] == 204
    assert np.isfinite(array).all()

print("All split-integrity checks passed.")
print("Coordinate overlap across splits: 0")

## 8. Save the Day 6 deliverables

In [ ]:
output_dir = "/content/TerraSpectra_Day06"
os.makedirs(output_dir, exist_ok=True)

np.save(os.path.join(output_dir, "X_train.npy"), X_train)
np.save(os.path.join(output_dir, "X_val.npy"), X_val)
np.save(os.path.join(output_dir, "X_test.npy"), X_test)

coord_train.to_csv(os.path.join(output_dir, "train_coordinates.csv"), index=False)
coord_val.to_csv(os.path.join(output_dir, "validation_coordinates.csv"), index=False)
coord_test.to_csv(os.path.join(output_dir, "test_coordinates.csv"), index=False)

split_report = pd.DataFrame([
    {"split": "train", "samples": len(X_train), "features": X_train.shape[1]},
    {"split": "validation", "samples": len(X_val), "features": X_val.shape[1]},
    {"split": "test", "samples": len(X_test), "features": X_test.shape[1]},
])
split_report.to_csv(os.path.join(output_dir, "dataset_split_report.csv"), index=False)

metadata = pd.DataFrame([{
    "cube_source": cube_source,
    "cube_height": cube.shape[0],
    "cube_width": cube.shape[1],
    "spectral_bands": cube.shape[2],
    "unique_annotated_coordinates": len(coordinates),
    "random_state": RANDOM_STATE,
    "split_ratio": "70:15:15",
    "labels_available": False,
    "note": "Feature split only; no disease class labels were supplied."
}])
metadata.to_csv(os.path.join(output_dir, "dataset_metadata.csv"), index=False)

display(split_report)
print("Saved files:", sorted(os.listdir(output_dir)))

## 9. Create a ZIP archive for backup / sharing

In [ ]:
zip_path = "/content/TerraSpectra_Day06_Outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in os.listdir(output_dir):
        full_path = os.path.join(output_dir, filename)
        archive.write(full_path, arcname=os.path.join("TerraSpectra_Day06", filename))

print("Created:", zip_path)
files.download(zip_path)

## 10. Completion summary and limitations

This notebook creates a reproducible feature split from the annotated coordinates. It does **not** train a disease classifier because the supplied coordinate CSV contains no verified class labels. Also, a random pixel split can place nearby pixels from the same plant in different subsets; for rigorous spatial generalization, a plant-/plot-level grouped split should be used once reliable group identifiers and labels are available.

**Suggested Git commit:**

`Day 6: Prepare reproducible train validation and test data splits`

**Suggested repository location:** `ml/notebooks/TerraSpectra_Day06_Train_Val_Test_Split.ipynb`